# Kriging with the Vecchia log-likelihood — objective="VLL(m)" (Octave)

This notebook demonstrates the **Vecchia approximated log-likelihood** as a fit
objective. Vecchia (1988): log p(y) is approximated by the product of the
conditionals of each observation given its $m$ nearest previously-ordered
neighbors (greedy maxmin ordering, Guinness 2018):

$$\log L \approx \sum_i \log p(y_i \mid y_{N(i)}), \qquad |N(i)| \le m$$

Each evaluation costs $O(n\,m^3)$ instead of $O(n^3)$, is a valid Gaussian
density (sparse inverse Cholesky), and is exact for $m = n-1$. The usage could
not be simpler — it is just an `objective` string:

- `objective="VLL"` (default $m=30$) or `objective="VLL(m)"`.

After the fit, ONE exact factorization is performed at $\theta^*$, so
`predict`/`simulate`/`update` behave exactly as after an `"LL"` fit.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 1500$)
4. Fit with `"VLL(30)"` vs exact `"LL"`: wall time and hyperparameters
5. Predictions: both models on a grid, RMSE
6. Effect of the number of neighbors $m$


## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [ ]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

## 2. Branin function

In [ ]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);
figure; contourf(G1, G2, z_true, 20); colorbar; title('True Branin');

## 3. Design of experiments (n = 1500)

In [ ]:
rand('seed', 123);
n = 1500;
X = rand(n, 2);
y = branin(X);

## 4. Fit: `VLL(30)` vs exact `LL`

In [ ]:
tic; k_vll = Kriging(y, X, "matern5_2", "none", [], "constant", false, "BFGS", "VLL(30)"); t_vll = toc;
tic; k_ll  = Kriging(y, X, "matern5_2", "none", [], "constant", false, "BFGS", "LL"); t_ll = toc;
printf('fit VLL(30) : %6.1f s   theta = %s\n', t_vll, mat2str(round(k_vll.theta()*1e4)/1e4));
printf('fit LL      : %6.1f s   theta = %s\n', t_ll, mat2str(round(k_ll.theta()*1e4)/1e4));
printf('speedup     : %.1fx\n', t_ll / t_vll);

## 5. Predictions

In [ ]:
m_vll = k_vll.predict(grid_pts);
m_ll = k_ll.predict(grid_pts);
figure; contourf(G1, G2, reshape(m_vll, 50, 50), 20); colorbar; title('mean — fit VLL(30)');
printf('RMSE vs true (VLL): %g\n', sqrt(mean((m_vll - z_true(:)).^2)));
printf('RMSE vs true (LL) : %g\n', sqrt(mean((m_ll - z_true(:)).^2)));
printf('max |VLL - LL|    : %g\n', max(abs(m_vll - m_ll)));

## 6. Effect of the number of neighbors $m$

In [ ]:
for m = [5 15 30]
    tic; k = Kriging(y, X, "matern5_2", "none", [], "constant", false, "BFGS", sprintf("VLL(%d)", m)); t = toc;
    mp = k.predict(grid_pts);
    printf('VLL(%2d) : fit %6.1f s   RMSE = %.4f\n', m, t, sqrt(mean((mp - z_true(:)).^2)));
end

## Notes

- The screening effect behind Vecchia weakens in high dimension: recommended
  for $d \le 5$ (complementary to `NestedKriging`, which is dimension-robust).
- The final exact factorization at $\theta^*$ still costs $O(n^3)$ once. For
  very large $n$, the C++ API offers `predictVecchia` (local $O(q\,m^3)$
  prediction) and a "light" mode (`set_vecchia_exact_commit(false)`) that
  skips it entirely — bindings for both are planned.
- References: Vecchia (1988); Guinness (2018), *Technometrics*; Katzfuss &
  Guinness (2021), *Statistical Science*.
